# Revision re-analyses: final retained paper output

This notebook retains the **in-domain class-wise F1 difference / bootstrap-CI analysis** that produced the source data for **Table 2** (and the associated confidence-interval reporting).

## Inputs

Four **linear-probe-stage** confusion matrices for the in-domain 17-class task:

- Baseline
- Fovea-Gaze
- Periph-NF
- Periph

## Output

`top_classes_delta_f1_with_95CI.csv`

- per-class F1 from aggregate confusion matrices;
- 1,000-replicate row-wise multinomial bootstrap;
- seed = 1337;
- comparisons:
  - Fovea-Gaze − Baseline
  - Fovea-Gaze − Periph-NF
  - Fovea-Gaze − Periph
- ranking by the largest absolute F1 difference across the three comparisons.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# Edit these paths if your released result files are stored elsewhere.
# These should point to the LINEAR-PROBE confusion matrices used for
# the reported in-domain analysis.

DATA_DIR = Path("results/in_domain")

PATHS = {
    "base":          DATA_DIR / "baseline_best_confusion_matrix.csv",
    "fovea":         DATA_DIR / "fovea_gaze_best_confusion_matrix.csv",
    "periph_nf":     DATA_DIR / "periph_nf_best_confusion_matrix.csv",
    "periph_non_nf": DATA_DIR / "periph_best_confusion_matrix.csv",
}

OUT_DIR = Path("outputs")
OUT_PATH = OUT_DIR / "top_classes_delta_f1_with_95CI.csv"

N_BOOT = 1000
SEED = 1337

In [ ]:
# ---------------------------------------------------------------------
# analysis logic
# ---------------------------------------------------------------------

def load_cm_csv(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Confusion matrix not found: {path}")

    df = pd.read_csv(path)

    true_col = df.columns[0]  # usually "true\\pred"
    true_labels = df[true_col].astype(str).tolist()
    pred_labels = [c for c in df.columns[1:]]
    mat = df.iloc[:, 1:].to_numpy(dtype=float)

    return true_labels, pred_labels, mat


def align_cms(cms):
    common_true = list(cms[next(iter(cms))][0])
    common_pred = list(cms[next(iter(cms))][1])

    for name, (t, p, m) in cms.items():
        if t != common_true or p != common_pred:
            df = pd.DataFrame(m, index=t, columns=p)
            df = df.loc[common_true, common_pred]
            cms[name] = (
                common_true,
                common_pred,
                df.to_numpy(dtype=float),
            )

    return cms, common_true


def f1_per_class_from_cm(cm):
    cm = np.asarray(cm, dtype=float)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    denom = 2 * tp + fp + fn

    return np.where(
        denom > 0,
        2 * tp / denom,
        np.nan,
    )


def simulate_cm_from_rows(cm, rng):
    # Parametric multinomial bootstrap
    # Each true-class row is resampled conditional on the observed row
    # total and observed predicted-class proportions.
    cm = np.asarray(cm, dtype=float)
    out = np.zeros_like(cm)

    row_sums = cm.sum(axis=1)

    for i in range(cm.shape[0]):
        n = int(round(row_sums[i]))

        if n <= 0:
            continue

        p = cm[i] / row_sums[i]
        p = np.clip(p, 0, 1)
        p = p / p.sum()

        out[i] = rng.multinomial(
            n=n,
            pvals=p,
        )

    return out


def bootstrap_delta_f1(cms, labels, a, b, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)

    cm_a = cms[a][2]
    cm_b = cms[b][2]

    f1_a_obs = f1_per_class_from_cm(cm_a)
    f1_b_obs = f1_per_class_from_cm(cm_b)
    delta_obs = f1_a_obs - f1_b_obs

    deltas = []

    for _ in range(n_boot):
        boot_a = simulate_cm_from_rows(cm_a, rng)
        boot_b = simulate_cm_from_rows(cm_b, rng)

        f1_a = f1_per_class_from_cm(boot_a)
        f1_b = f1_per_class_from_cm(boot_b)

        deltas.append(f1_a - f1_b)

    deltas = np.stack(deltas, axis=0)

    ci_low = np.nanpercentile(deltas, 2.5, axis=0)
    ci_high = np.nanpercentile(deltas, 97.5, axis=0)

    return pd.DataFrame(
        {
            "Label": labels,
            "delta_obs": delta_obs,
            "ci_low": ci_low,
            "ci_high": ci_high,
        }
    )


def fmt_num(x, ndigits=3):
    if pd.isna(x):
        return ""
    return f"{x:.{ndigits}f}"


def fmt_ci(lo, hi, ndigits=3):
    if pd.isna(lo) or pd.isna(hi):
        return ""
    return f"[{lo:.{ndigits}f}, {hi:.{ndigits}f}]"


def pretty_minus(s):
    return str(s).replace("-", "−")

In [ ]:
# ---------------------------------------------------------------------
# Compute the three comparisons and build the final table
# ---------------------------------------------------------------------

cms = {
    name: load_cm_csv(path)
    for name, path in PATHS.items()
}

cms, labels = align_cms(cms)

if len(labels) != 17:
    raise ValueError(
        f"Expected the in-domain 17-class confusion matrices; found {len(labels)} labels."
    )

ci_fovea_base = bootstrap_delta_f1(
    cms,
    labels,
    a="fovea",
    b="base",
    n_boot=N_BOOT,
    seed=SEED,
)

ci_fovea_nf = bootstrap_delta_f1(
    cms,
    labels,
    a="fovea",
    b="periph_nf",
    n_boot=N_BOOT,
    seed=SEED,
)

ci_fovea_p = bootstrap_delta_f1(
    cms,
    labels,
    a="fovea",
    b="periph_non_nf",
    n_boot=N_BOOT,
    seed=SEED,
)


tbl = (
    ci_fovea_base.rename(
        columns={
            "delta_obs": "Fovea > Base",
            "ci_low": "Fovea > Base_low",
            "ci_high": "Fovea > Base_high",
        }
    )
    .merge(
        ci_fovea_nf.rename(
            columns={
                "delta_obs": "Fovea > Periph-NF",
                "ci_low": "Fovea > Periph-NF_low",
                "ci_high": "Fovea > Periph-NF_high",
            }
        ),
        on="Label",
        how="outer",
    )
    .merge(
        ci_fovea_p.rename(
            columns={
                "delta_obs": "Fovea > Periph",
                "ci_low": "Fovea > Periph_low",
                "ci_high": "Fovea > Periph_high",
            }
        ),
        on="Label",
        how="outer",
    )
)


# max_j |Delta_j|
delta_cols = [
    "Fovea > Base",
    "Fovea > Periph-NF",
    "Fovea > Periph",
]

tbl["max_abs_delta"] = tbl[delta_cols].abs().max(axis=1)

tbl = (
    tbl.sort_values(
        "max_abs_delta",
        ascending=False,
    )
    .reset_index(drop=True)
)

tbl.insert(
    0,
    "Rank",
    np.arange(1, len(tbl) + 1),
)


# Confidence-interval strings
tbl["95% CI (Base)"] = [
    fmt_ci(lo, hi)
    for lo, hi in zip(
        tbl["Fovea > Base_low"],
        tbl["Fovea > Base_high"],
    )
]

tbl["95% CI (Periph-NF)"] = [
    fmt_ci(lo, hi)
    for lo, hi in zip(
        tbl["Fovea > Periph-NF_low"],
        tbl["Fovea > Periph-NF_high"],
    )
]

tbl["95% CI (Periph)"] = [
    fmt_ci(lo, hi)
    for lo, hi in zip(
        tbl["Fovea > Periph_low"],
        tbl["Fovea > Periph_high"],
    )
]


# Final formatting
for c in delta_cols:
    tbl[c] = tbl[c].map(fmt_num)

for c in (
    delta_cols
    + [
        "95% CI (Base)",
        "95% CI (Periph-NF)",
        "95% CI (Periph)",
    ]
):
    tbl[c] = tbl[c].map(pretty_minus)


final_table = tbl[
    [
        "Rank",
        "Label",
        "Fovea > Base",
        "95% CI (Base)",
        "Fovea > Periph-NF",
        "95% CI (Periph-NF)",
        "Fovea > Periph",
        "95% CI (Periph)",
    ]
]

display(final_table)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

final_table.to_csv(
    OUT_PATH,
    index=False,
)

print(f"Saved CSV to: {OUT_PATH}")

## Published-output sanity check

With the original paper confusion matrices, the first rows should begin with:

1. `playing video game`
2. `solving rubik's cube`
3. `socializing`
4. `computer work`

The exact reported F1 differences in the paper begin with approximately:

- `playing video game`: 0.433 / 0.433 / 0.433
- `solving rubik's cube`: −0.116 / 0.428 / 0.018
- `socializing`: 0.365 / 0.318 / 0.336

Note* This cell does not alter the analysis; it only checks that the supplied confusion matrices correspond to the intended final linear-probe artifacts.

In [ ]:
expected_top = [
    "playing video game",
    "solving rubik's cube",
    "socializing",
    "computer work",
]

observed_top = (
    final_table["Label"]
    .astype(str)
    .str.lower()
    .tolist()[:4]
)

if observed_top != expected_top:
    print(
        "[WARN] The top-ranked labels do not match the published final Table 2. "
        "Check that PATHS points to the correct LINEAR-PROBE confusion matrices."
    )
else:
    print("[OK] Top-ranked labels match the published final Table 2 ordering.")